# U-Net from scratch — Semantic Segmentation on FashionMNIST

**Semantic segmentation** is the task of assigning a label to **every pixel** of an image. Where a classifier outputs one label per image, a segmentation model outputs a label map of the same spatial size as the input.

## Why U-Net?

U-Net (Ronneberger et al., 2015) is the standard starting point for segmentation. It has two key design ideas:

1. **Encoder–decoder with a bottleneck.** The encoder progressively downsamples the image, capturing high-level *"what"*. The decoder upsamples it back, recovering the spatial detail *"where"*.
2. **Skip connections.** At every level the encoder feature map is concatenated into the matching decoder block. This gives the decoder direct access to high-resolution features that would otherwise be lost in the bottleneck.

```
            ENCODER                                  DECODER
 input → [DoubleConv] ──────────────── skip ────────────── [DoubleConv] → 1×1 Conv → mask
              │                                                  ↑
           MaxPool                                           ConvTranspose
              ↓                                                  │
        [DoubleConv] ────────────── skip ─────────────── [DoubleConv]
              │                                                  ↑
           MaxPool                                           ConvTranspose
              ↓                                                  │
                     ──────── [DoubleConv] (bottleneck) ────────
```

## This notebook

1. Build a **synthetic segmentation dataset** by thresholding FashionMNIST images — each garment becomes a foreground mask.
2. Implement **U-Net from scratch** in PyTorch, with shape comments throughout.
3. Train it with **Dice loss** and evaluate with **IoU**, two standard segmentation metrics.
4. Visualize the predicted masks against ground truth.

## 1. Setup

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torchvision
from torchvision import transforms

# Auto-detect device: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. A synthetic segmentation dataset

FashionMNIST comes with **classification labels** (shirt, bag, sneaker, ...), not segmentation masks. We create a pseudo-segmentation task by **thresholding the pixel intensity**: any pixel brighter than `0.1` is declared "foreground" (the garment), everything else is "background".

This gives us clean `(image, mask)` pairs where both tensors have shape `(1, 28, 28)` and mask values are in `{0, 1}`. It is a **toy task** — the ground truth is computable from the input — but it is perfect for teaching because:

- The task has a well-defined solution (you could replicate it without learning).
- Training is fast (28×28 images).
- Qualitative evaluation is obvious: the predicted mask should look like the input silhouette.

In [ ]:
class FashionSegmentationDataset(Dataset):
    """Wraps FashionMNIST and returns (image, binary_mask) pairs.

    The mask is produced on the fly by thresholding pixel intensity:
        mask = 1  if pixel > threshold else 0
    """

    def __init__(self, train=True, threshold=0.1):
        self.data = torchvision.datasets.FashionMNIST(
            root="./data", train=train, download=True, transform=transforms.ToTensor(),
        )
        self.threshold = threshold

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, _ = self.data[idx]                        # (1, 28, 28) float in [0, 1]
        mask = (img > self.threshold).float()          # (1, 28, 28) float in {0, 1}
        return img, mask

### Visualize a few samples

In [ ]:
def visualize_dataset_samples(dataset, num_samples=5):
    """Show `num_samples` (image, mask) pairs side by side."""
    fig, axes = plt.subplots(num_samples, 2, figsize=(5, 2 * num_samples))
    for i in range(num_samples):
        image, mask = dataset[i]
        axes[i, 0].imshow(image.squeeze(), cmap="gray")
        axes[i, 0].set_title("Input image")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(mask.squeeze(), cmap="gray")
        axes[i, 1].set_title("Pseudo mask")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.show()


full_train = FashionSegmentationDataset(train=True)
full_test  = FashionSegmentationDataset(train=False)

visualize_dataset_samples(full_train, num_samples=5)

### Subsample and build train / validation loaders

We use 20% of the training set to keep training fast, and further split it 90/10 into train and validation so we can monitor generalization instead of only looking at training loss.

In [ ]:
rng = np.random.default_rng(SEED)
subset_indices = rng.choice(len(full_train), size=int(0.2 * len(full_train)), replace=False)
full_subset = Subset(full_train, subset_indices.tolist())

val_size   = int(0.1 * len(full_subset))
train_size = len(full_subset) - val_size
train_ds, val_ds = random_split(
    full_subset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}   Val: {len(val_ds)}")

## 3. Segmentation metrics — Dice loss and IoU

Why not just use per-pixel cross-entropy? Because segmentation masks are usually **very imbalanced** — most pixels are background. A pixelwise loss can be minimized by predicting "background" everywhere. We need metrics that focus on the **overlap between prediction and ground truth**.

### Dice coefficient

$$
\mathrm{Dice}(P, G) \; = \; \frac{2 \, |P \cap G|}{|P| + |G|}
$$

- `P` = predicted mask, `G` = ground truth mask, `| \cdot |` = sum of pixel values.
- Ranges from 0 (no overlap) to 1 (perfect match).
- We minimize `1 - Dice` as the loss. A `smooth` constant in numerator and denominator avoids division by zero and stabilizes gradients when both masks are nearly empty.

### Intersection over Union (IoU / Jaccard)

$$
\mathrm{IoU}(P, G) \; = \; \frac{|P \cap G|}{|P \cup G|} \; = \; \frac{|P \cap G|}{|P| + |G| - |P \cap G|}
$$

- Also in `[0, 1]`, strictly more punishing than Dice (IoU ≤ Dice).
- We use it as a **monitoring metric** — it does not need to be differentiable because we only evaluate it, we don't backprop through it.

In [ ]:
class DiceLoss(nn.Module):
    """Differentiable Dice loss for binary segmentation.

    Expects `preds` to be probabilities in [0, 1] (apply sigmoid first) and
    `targets` in {0, 1}, both of shape (B, 1, H, W).
    """

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds   = preds.contiguous()
        targets = targets.contiguous()

        # Reduce over spatial dims; keep batch and channel separate
        intersection = (preds * targets).sum(dim=(2, 3))
        denom        = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))

        dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()


@torch.no_grad()
def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    """Mean IoU over a batch. Inputs expected as (B, 1, H, W) probabilities and {0,1} masks."""
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum(dim=(2, 3))
    union        = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

## 4. The U-Net architecture

We build U-Net out of a reusable `DoubleConv` block, then stack two encoder levels, a bottleneck, and two decoder levels. Every decoder step *concatenates* the matching encoder feature map — that's the skip connection.

### Shape trace (input 1 × 28 × 28)

| Stage         | Op                                   | Output channels | Spatial |
|---------------|--------------------------------------|-----------------|---------|
| `enc1`        | DoubleConv(1 → 32)                   | 32              | 28×28   |
| `pool1`       | MaxPool(2)                           | 32              | 14×14   |
| `enc2`        | DoubleConv(32 → 64)                  | 64              | 14×14   |
| `pool2`       | MaxPool(2)                           | 64              | 7×7     |
| `bottleneck`  | DoubleConv(64 → 128)                 | 128             | 7×7     |
| `up2`         | ConvTranspose(128 → 64, stride 2)    | 64              | 14×14   |
| `dec2`        | DoubleConv(64+64 → 64)               | 64              | 14×14   |
| `up1`         | ConvTranspose(64 → 32, stride 2)     | 32              | 28×28   |
| `dec1`        | DoubleConv(32+32 → 32)               | 32              | 28×28   |
| `final`       | Conv 1×1 (32 → 1) + Sigmoid          | 1               | 28×28   |

> **Notice** the `64+64` and `32+32` in the decoder DoubleConvs — that's the channel dimension *after concatenating the skip connection*.

In [ ]:
class DoubleConv(nn.Module):
    """Two Conv3x3 + ReLU in a row — the basic building block of U-Net."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels,  out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Minimal U-Net for 1-channel 28x28 inputs and 1-channel binary masks."""

    def __init__(self, in_channels=1, out_channels=1, base=32):
        super().__init__()

        # Encoder
        self.enc1  = DoubleConv(in_channels, base)       # -> (B, base,     28, 28)
        self.pool1 = nn.MaxPool2d(2)                     # -> (B, base,     14, 14)
        self.enc2  = DoubleConv(base, base * 2)          # -> (B, 2*base,   14, 14)
        self.pool2 = nn.MaxPool2d(2)                     # -> (B, 2*base,    7,  7)

        # Bottleneck
        self.bottleneck = DoubleConv(base * 2, base * 4) # -> (B, 4*base,    7,  7)

        # Decoder
        self.up2  = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)  # -> 14x14
        self.dec2 = DoubleConv(base * 4, base * 2)       # in = 2*base (up) + 2*base (skip)
        self.up1  = nn.ConvTranspose2d(base * 2, base,     kernel_size=2, stride=2)  # -> 28x28
        self.dec1 = DoubleConv(base * 2, base)           # in = base (up) + base (skip)

        # 1x1 conv reduces to `out_channels` without changing spatial size
        self.final = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder — remember `e1` and `e2` for the skip connections
        e1 = self.enc1(x)                     # (B, base,    28, 28)
        e2 = self.enc2(self.pool1(e1))        # (B, 2*base,  14, 14)

        # Bottleneck
        b  = self.bottleneck(self.pool2(e2))  # (B, 4*base,   7,  7)

        # Decoder — upsample, concat with matching encoder output, conv
        d2 = self.up2(b)                                         # (B, 2*base, 14, 14)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))               # concat along channels

        d1 = self.up1(d2)                                        # (B, base,   28, 28)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.final(d1))                     # probabilities in [0, 1]

In [ ]:
model = UNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

## 5. Training

Per batch:
1. Forward the images through the U-Net → predicted mask probabilities.
2. Compute **Dice loss** against the ground-truth mask, backprop through the student.
3. Compute **IoU** as a non-differentiable monitor.

We track per-epoch train loss/IoU, val loss/IoU, and per-epoch training time plus the total time.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Run one pass over `loader`. If `optimizer` is None -> eval/no-grad mode.

    Returns
    -------
    avg_loss, avg_iou  — both averaged over samples.
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    running_iou  = 0.0
    n_samples = 0
    n_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss  = criterion(preds, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = imgs.size(0)
            running_loss += loss.item() * bs
            running_iou  += iou_score(preds, masks) * bs
            n_samples    += bs
            n_batches    += 1

    return running_loss / n_samples, running_iou / n_samples

In [ ]:
NUM_EPOCHS = 5

criterion = DiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss_hist, train_iou_hist = [], []
val_loss_hist,   val_iou_hist   = [], []
epoch_times = []

total_start = time.time()
for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    train_loss, train_iou = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_iou   = run_epoch(model, val_loader,   criterion, optimizer=None)

    epoch_time = time.time() - epoch_start
    train_loss_hist.append(train_loss); train_iou_hist.append(train_iou)
    val_loss_hist.append(val_loss);     val_iou_hist.append(val_iou)
    epoch_times.append(epoch_time)

    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}   "
          f"train dice: {train_loss:.4f}  iou: {train_iou:.4f}   |   "
          f"val dice: {val_loss:.4f}  iou: {val_iou:.4f}   "
          f"time: {epoch_time:.1f}s")

total_time = time.time() - total_start
print(f"\nTotal training time: {total_time:.1f}s   "
      f"(mean per epoch: {total_time / NUM_EPOCHS:.1f}s)")

### Loss and IoU curves

In [ ]:
epochs_x = np.arange(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_x, train_loss_hist, marker="o", label="train")
axes[0].plot(epochs_x, val_loss_hist,   marker="o", label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("Dice loss")
axes[0].set_title("Dice loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, train_iou_hist, marker="o", label="train")
axes[1].plot(epochs_x, val_iou_hist,   marker="o", label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("IoU")
axes[1].set_title("IoU"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].bar(epochs_x, epoch_times)
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("seconds")
axes[2].set_title(f"Time per epoch (total: {total_time:.1f}s)")
axes[2].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 6. Visualize predictions

For a few validation images we show: the input, the pseudo ground-truth mask, and the model's prediction (thresholded at 0.5). Since our ground-truth *is* a threshold on the input, an ideal U-Net would reproduce the mask exactly.

In [ ]:
def visualize_results(model, dataset, num_samples=5, threshold=0.5):
    """Plot input / ground-truth / prediction triples."""
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(9, 2 * num_samples))

    with torch.no_grad():
        for i in range(num_samples):
            image, mask = dataset[i]
            pred = model(image.unsqueeze(0).to(device))
            pred_mask = (pred.squeeze().cpu().numpy() > threshold).astype(float)

            axes[i, 0].imshow(image.squeeze(), cmap="gray");    axes[i, 0].set_title("Input")
            axes[i, 1].imshow(mask.squeeze(),  cmap="gray");    axes[i, 1].set_title("Ground truth")
            axes[i, 2].imshow(pred_mask,       cmap="gray");    axes[i, 2].set_title("Prediction")
            for j in range(3):
                axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()


visualize_results(model, val_ds, num_samples=5)

## 7. Takeaways and things to try next

- **U-Net = encoder + decoder + skip connections.** The skips are what let the decoder recover spatial detail that pooling threw away.
- **Dice loss / IoU** are the right metrics for segmentation because they focus on mask *overlap* rather than per-pixel accuracy — crucial when foreground is rare.
- Our task is synthetic — the ground truth is a function of the input — so very high IoU is expected and not a sign of a general-purpose segmenter.

### Exercises

1. **Real masks.** Swap FashionMNIST for a dataset with actual segmentation labels (e.g. the Oxford Pets segmentation dataset or a synthetic shapes dataset). How does IoU change?
2. **Deeper U-Net.** Add a third encoder/decoder level (64 → 128 → 256 → bottleneck 512). Does training still converge? What happens to per-epoch time?
3. **Change the base width.** Try `base=16` and `base=64` in the `UNet` constructor and compare accuracy vs parameter count vs training time.
4. **Multi-class segmentation.** Remove the `sigmoid` at the end, set `out_channels = num_classes`, and train with pixelwise cross-entropy to segment several classes at once.
5. **Combine losses.** Many real systems use `Dice + BCE` as the loss. Add a BCE term, tune its weight, and see whether training becomes more stable.